In [1]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────
# LOAD ALL FILES
# ─────────────────────────────────────────
# Kathmandu — two files to merge
ktm_fixed = pd.read_csv('lalpurja_house_kathmandu_fixed.csv')
ktm_stats = pd.read_csv('lalpurja_house_kathmandu_stats.csv')

# Lalitpur and Bhaktapur — stats files have everything
lat_stats = pd.read_csv('lalpurja_house_lalitpur_stats.csv')
bkt_stats = pd.read_csv('lalpurja_house_bhaktapur_stats.csv')

print("=== Kathmandu Fixed ===")
print(f"Shape: {ktm_fixed.shape}")
print(f"Columns: {list(ktm_fixed.columns)}")

print("\n=== Kathmandu Stats ===")
print(f"Shape: {ktm_stats.shape}")
print(f"New cols in stats: {[c for c in ktm_stats.columns if c not in ktm_fixed.columns]}")

print("\n=== Common key to merge on ===")
print(f"property_id in fixed: {'property_id' in ktm_fixed.columns}")
print(f"property_id in stats: {'property_id' in ktm_stats.columns}")

# Check if property_id is a reliable key
print(f"\nFixed unique property_ids: {ktm_fixed['property_id'].nunique()}")
print(f"Stats unique property_ids: {ktm_stats['property_id'].nunique()}")
print(f"\nSample property_ids fixed: {ktm_fixed['property_id'].head(5).tolist()}")
print(f"Sample property_ids stats: {ktm_stats['property_id'].head(5).tolist()}")

=== Kathmandu Fixed ===
Shape: (1605, 43)
Columns: ['url', 'source', 'category', 'district', 'property_id', 'title', 'total_price_raw', 'property_type', 'property_face', 'year_built', 'road_type', 'negotiable', 'furnishing', 'city_area', 'pricing_raw', 'purpose', 'property_area', 'built_up_area', 'road_access', 'dimension', 'date_posted', 'views', 'municipality', 'ward_no', 'ring_road', 'landmark', 'hospital', 'airport', 'pharmacy', 'bhatbhateni', 'school', 'college', 'gym', 'public_transport', 'police_station', 'pashupatinath', 'boudhanath', 'atm', 'hotel', 'restaurant', 'banquet', 'ward_office', 'neighborhood']

=== Kathmandu Stats ===
Shape: (1605, 51)
New cols in stats: ['property_type_stat', 'land_area_aana', 'bedrooms', 'kitchens', 'bathrooms', 'living_rooms', 'parking', 'total_floors']

=== Common key to merge on ===
property_id in fixed: True
property_id in stats: True

Fixed unique property_ids: 1605
Stats unique property_ids: 1605

Sample property_ids fixed: [8573, 8559, 8558

In [2]:
# ─────────────────────────────────────────
# STEP 1 — MERGE KATHMANDU FILES
# Both have exactly 1605 rows with matching property_ids
# Extract only new columns from stats file
# Merge on property_id — reliable unique key
# ─────────────────────────────────────────

# New columns to extract from stats file
new_cols = ['property_id', 'bedrooms', 'kitchens',
            'bathrooms', 'living_rooms', 'parking', 'total_floors']

ktm_new_cols = ktm_stats[new_cols]

# Merge fixed file with new columns
ktm_merged = ktm_fixed.merge(ktm_new_cols, on='property_id', how='left')

print(f"=== Kathmandu Merged ===")
print(f"Shape: {ktm_merged.shape}")
print(f"Columns: {list(ktm_merged.columns)}")
print(f"\nNew columns null check:")
for col in ['bedrooms', 'kitchens', 'bathrooms', 'living_rooms', 'parking', 'total_floors']:
    print(f"  {col}: {ktm_merged[col].isnull().sum()} nulls")
print(f"\nSample new columns:")
print(ktm_merged[['property_id', 'bedrooms', 'bathrooms',
                   'total_floors', 'parking']].head(5))

# ─────────────────────────────────────────
# STEP 2 — CHECK LALITPUR AND BHAKTAPUR STATS
# These already have amenities + new columns
# Just need to verify column consistency
# ─────────────────────────────────────────
print(f"\n=== Lalitpur Stats ===")
print(f"Shape: {lat_stats.shape}")
print(f"Has bedrooms: {'bedrooms' in lat_stats.columns}")
print(f"Bedrooms nulls: {lat_stats['bedrooms'].isnull().sum()}")

print(f"\n=== Bhaktapur Stats ===")
print(f"Shape: {bkt_stats.shape}")
print(f"Has bedrooms: {'bedrooms' in bkt_stats.columns}")
print(f"Bedrooms nulls: {bkt_stats['bedrooms'].isnull().sum()}")

# Check if all three have same columns
ktm_cols = set(ktm_merged.columns)
lat_cols = set(lat_stats.columns)
bkt_cols = set(bkt_stats.columns)

print(f"\n=== Column differences ===")
print(f"In Kathmandu but not Lalitpur: {ktm_cols - lat_cols}")
print(f"In Lalitpur but not Kathmandu: {lat_cols - ktm_cols}")
print(f"In Kathmandu but not Bhaktapur: {ktm_cols - bkt_cols}")
print(f"In Bhaktapur but not Kathmandu: {bkt_cols - ktm_cols}")

=== Kathmandu Merged ===
Shape: (1605, 49)
Columns: ['url', 'source', 'category', 'district', 'property_id', 'title', 'total_price_raw', 'property_type', 'property_face', 'year_built', 'road_type', 'negotiable', 'furnishing', 'city_area', 'pricing_raw', 'purpose', 'property_area', 'built_up_area', 'road_access', 'dimension', 'date_posted', 'views', 'municipality', 'ward_no', 'ring_road', 'landmark', 'hospital', 'airport', 'pharmacy', 'bhatbhateni', 'school', 'college', 'gym', 'public_transport', 'police_station', 'pashupatinath', 'boudhanath', 'atm', 'hotel', 'restaurant', 'banquet', 'ward_office', 'neighborhood', 'bedrooms', 'kitchens', 'bathrooms', 'living_rooms', 'parking', 'total_floors']

New columns null check:
  bedrooms: 172 nulls
  kitchens: 240 nulls
  bathrooms: 229 nulls
  living_rooms: 350 nulls
  parking: 343 nulls
  total_floors: 134 nulls

Sample new columns:
   property_id  bedrooms  bathrooms  total_floors  parking
0         8573       5.0        5.0           2.5    

In [3]:
# ─────────────────────────────────────────
# STEP 3 — COMBINE ALL THREE DISTRICTS
# All files now have identical 49 columns
# Assign correct district names before concat
# ─────────────────────────────────────────
ktm_merged['district'] = 'Kathmandu'
lat_stats['district']  = 'Lalitpur'
bkt_stats['district']  = 'Bhaktapur'

df_new = pd.concat([ktm_merged, lat_stats, bkt_stats], ignore_index=True)

print(f"=== Combined Dataset ===")
print(f"Shape: {df_new.shape}")
print(f"Districts: {df_new['district'].value_counts().to_dict()}")

print(f"\n=== New columns null summary ===")
new_cols = ['bedrooms', 'kitchens', 'bathrooms',
            'living_rooms', 'parking', 'total_floors']
for col in new_cols:
    nulls = df_new[col].isnull().sum()
    pct   = nulls / len(df_new) * 100
    print(f"  {col}: {nulls} nulls ({pct:.1f}%)")

print(f"\n=== Sample distributions ===")
for col in new_cols:
    print(f"\n{col}:")
    print(df_new[col].value_counts().head(8))

=== Combined Dataset ===
Shape: (2328, 49)
Districts: {'Kathmandu': 1605, 'Lalitpur': 539, 'Bhaktapur': 184}

=== New columns null summary ===
  bedrooms: 204 nulls (8.8%)
  kitchens: 275 nulls (11.8%)
  bathrooms: 262 nulls (11.3%)
  living_rooms: 405 nulls (17.4%)
  parking: 396 nulls (17.0%)
  total_floors: 170 nulls (7.3%)

=== Sample distributions ===

bedrooms:
bedrooms
6.0    531
5.0    464
4.0    292
7.0    207
8.0    138
3.0    108
2.0     59
9.0     57
Name: count, dtype: int64

kitchens:
kitchens
2.0    930
1.0    729
3.0    306
4.0     48
5.0     29
6.0      6
7.0      4
8.0      1
Name: count, dtype: int64

bathrooms:
bathrooms
4.0    642
3.0    503
5.0    438
6.0    141
2.0    128
7.0     76
1.0     60
8.0     39
Name: count, dtype: int64

living_rooms:
living_rooms
2.0    906
1.0    688
3.0    273
4.0     33
5.0     15
6.0      5
8.0      2
7.0      1
Name: count, dtype: int64

parking:
parking
1.0     754
2.0     541
3.0     264
4.0     150
5.0     131
6.0      35
10.0 

In [5]:
df_new.sample(12)

,url,source,category,district,property_id,title,total_price_raw,property_type,property_face,year_built,...,restaurant,banquet,ward_office,neighborhood,bedrooms,kitchens,bathrooms,living_rooms,parking,total_floors
1183,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Kathmandu,3770,Property Details,2 Crore 70 Lakh,Residential,South,NaN,...,50m,50m,150m,Budhanilkantha,6.0,2.0,4.0,1.0,1.0,2.5
2062,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Lalitpur,3299,Property Details,2 Crore 60 Lakh,Residential,North-East,2024,...,200m,200m,500m,Imadol,5.0,2.0,4.0,1.0,3.0,2.5
1167,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Kathmandu,3840,Property Details,1 Crore 50 Lakh,Residential,North-West,2081,...,NaN,NaN,1.5km,Tarkeshwor,5.0,1.0,4.0,2.0,1.0,2.5
1377,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Kathmandu,2937,Property Details,2 Crore 80 Lakh,Residential,North-East,NaN,...,few minutes walking distance,few minutes walking distance,NaN,Nagarjun,8.0,2.0,4.0,3.0,2.0,3.0
2276,https://lalpurjanepal.com.np/properties/semi-c...,lalpurja,house,Bhaktapur,4738,Property Details,6 Crore 15 Lakh,Semi-commercial,West,2021,...,NaN,NaN,1.5km,Madhyapur Thimi,20.0,7.0,16.0,8.0,2.0,5.5
1863,https://lalpurjanepal.com.np/properties/land-o...,lalpurja,house,Lalitpur,5388,Property Details,40 Lakh Per Aana,Residential,West,NaN,...,200m,1.5km,500m,Imadol,NaN,NaN,NaN,NaN,NaN,NaN
1442,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Kathmandu,2771,Property Details,3 Crore 49 Lakh,Residential,North-East,NaN,...,NaN,NaN,32,Koteshwor,9.0,2.0,6.0,2.0,NaN,NaN
1718,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Lalitpur,7293,Property Details,3 Crore 40 Lakh,Residential,South,2079,...,170m,300m,01,Imadol,6.0,2.0,5.0,2.0,1.0,2.5
1156,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Kathmandu,3863,Property Details,2 Crore 5 Lakh,Residential,NaN,2062,...,NaN,NaN,1km,Kapan,7.0,2.0,3.0,1.0,1.0,2.5
2282,https://lalpurjanepal.com.np/properties/house-...,lalpurja,house,Bhaktapur,4553,Property Details,2 Crore 15 Lakh,Residential,West,2022,...,200m,400m,400m,Gatthaghar,3.0,2.0,3.0,2.0,1.0,2.5


In [6]:
import re
from datetime import datetime

# ─────────────────────────────────────────
# STEP 1 — DROP USELESS COLUMNS
# Same as previous cleaning pipeline
# ─────────────────────────────────────────
drop_cols = ['url', 'source', 'property_id', 'title',
             'purpose', 'city_area', 'category',
             'landmark', 'ward_office', 'pricing_raw']
df_new.drop(columns=drop_cols, inplace=True)
print(f"Shape after dropping useless cols: {df_new.shape}")

# ─────────────────────────────────────────
# STEP 2 — PARSE total_price_raw → total_price
# ─────────────────────────────────────────
def parse_nepali_price(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    total = 0
    crore_match = re.search(r'(\d+\.?\d*)\s*crore', val)
    if crore_match: total += float(crore_match.group(1)) * 10_000_000
    lakh_match = re.search(r'(\d+\.?\d*)\s*lakh', val)
    if lakh_match: total += float(lakh_match.group(1)) * 100_000
    return total if total > 0 else np.nan

df_new['total_price'] = df_new['total_price_raw'].apply(parse_nepali_price)
print(f"total_price nulls: {df_new['total_price'].isnull().sum()}")
print(f"Price range: {df_new['total_price'].min():,.0f} – {df_new['total_price'].max():,.0f}")

# ─────────────────────────────────────────
# STEP 3 — FILTER PRICE RANGE
# Min 10M (1 Crore) — minimum realistic house price
# Max 500M (50 Crore) — remove extreme outliers
# ─────────────────────────────────────────
before = len(df_new)
df_new = df_new[
    (df_new['total_price'] >= 10_000_000) &
    (df_new['total_price'] <= 500_000_000)
].reset_index(drop=True)
print(f"\nRows removed by price filter: {before - len(df_new)}")
print(f"Remaining rows: {len(df_new)}")

# ─────────────────────────────────────────
# STEP 4 — PARSE property_area → land_size_aana
# ─────────────────────────────────────────
def parse_property_area(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    num_match = re.search(r'(\d+\.?\d*)', val)
    if not num_match: return np.nan
    num = float(num_match.group(1))
    if 'aana' in val or 'anna' in val: return num
    elif 'ropani' in val: return num * 16
    elif 'paisa' in val: return round(num / 4, 3)
    elif 'daam' in val: return round(num / 16, 3)
    elif 'sq. meter' in val or 'sq meter' in val: return round(num / 31.8, 2)
    elif 'sq' in val or 'feet' in val or 'ft' in val: return round(num / 342.25, 2)
    return np.nan

df_new['land_size_aana'] = df_new['property_area'].apply(parse_property_area)
df_new['land_size_aana'] = df_new['land_size_aana'].clip(lower=1, upper=50)
print(f"\nland_size_aana nulls: {df_new['land_size_aana'].isnull().sum()}")
print(f"Range: {df_new['land_size_aana'].min()} – {df_new['land_size_aana'].max()} aana")

# ─────────────────────────────────────────
# STEP 5 — PARSE built_up_area → built_up_sqft
# ─────────────────────────────────────────
def parse_built_up_area(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    num_match = re.search(r'(\d+\.?\d*)', val)
    if not num_match: return np.nan
    num = float(num_match.group(1))
    if 'aana' in val or 'anna' in val:
        return num if num > 50 else round(num * 342.25, 2)
    elif 'ropani' in val: return round(num * 16 * 342.25, 2)
    elif 'paisa' in val: return round((num / 4) * 342.25, 2)
    elif 'sq. meter' in val or 'sq meter' in val: return round(num * 10.764, 2)
    elif 'square-feet' in val or 'sq' in val or 'feet' in val or 'ft' in val: return num
    else: return num

df_new['built_up_sqft'] = df_new['built_up_area'].apply(parse_built_up_area)
print(f"\nbuilt_up_sqft nulls: {df_new['built_up_sqft'].isnull().sum()}")
print(f"Range: {df_new['built_up_sqft'].min():.0f} – {df_new['built_up_sqft'].max():.0f} sqft")

# ─────────────────────────────────────────
# STEP 6 — FIX year_built → house_age
# ─────────────────────────────────────────
def parse_year_built(val):
    if pd.isna(val): return np.nan
    val = str(val).strip()
    val_clean = val.replace(' ', '').upper()
    if '+' in val_clean: return np.nan
    if len(re.sub(r'[^0-9]', '', val_clean)) < 4: return np.nan
    if val_clean in ['20222']: return np.nan
    num_match = re.search(r'(\d{4})', val_clean)
    if not num_match: return np.nan
    year = int(num_match.group(1))
    if 'AD' in val_clean or 'A.D' in val_clean: return year
    if 'BS' in val_clean or 'B.S' in val_clean or 'AS' in val_clean: return year - 56
    if year > 2025: return year - 56
    elif 1950 <= year <= 2025: return year
    else: return np.nan

current_year = datetime.now().year
df_new['year_built_ad'] = df_new['year_built'].apply(parse_year_built)
df_new['year_built_ad'] = df_new['year_built_ad'].clip(upper=current_year)
df_new['house_age']     = current_year - df_new['year_built_ad']
print(f"\nhouse_age nulls: {df_new['house_age'].isnull().sum()}")
print(f"Range: {df_new['house_age'].min()} – {df_new['house_age'].max()} years")

# ─────────────────────────────────────────
# STEP 7 — PARSE road_access → road_width_feet
# ─────────────────────────────────────────
def parse_road_access(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    num_match = re.search(r'(\d+\.?\d*)', val)
    if not num_match: return np.nan
    num = float(num_match.group(1))
    return round(num * 3.281, 1) if 'meter' in val else num

df_new['road_width_feet'] = df_new['road_access'].apply(parse_road_access)
df_new['road_width_feet'] = df_new['road_width_feet'].clip(upper=80)
print(f"\nroad_width_feet nulls: {df_new['road_width_feet'].isnull().sum()}")

# ─────────────────────────────────────────
# STEP 8 — STANDARDIZE road_type
# ─────────────────────────────────────────
road_map = {
    'Blacktopped': 'High Access', 'Paved': 'High Access',
    'Gravelled':   'Low Access',  'Soil Stabilized': 'Low Access',
    'alley':       'Low Access'
}
df_new['road_type'] = df_new['road_type'].replace(road_map)
print(f"\nroad_type distribution:")
print(df_new['road_type'].value_counts())

# ─────────────────────────────────────────
# STEP 9 — CONVERT DISTANCE COLUMNS TO METERS
# ─────────────────────────────────────────
def convert_to_meters(val):
    if pd.isna(val): return np.nan
    val = str(val).strip().lower()
    num_match = re.search(r'(\d+\.?\d*)', val)
    if not num_match: return np.nan
    num = float(num_match.group(1))
    return num * 1000 if 'km' in val else num

distance_cols = ['hospital', 'airport', 'pharmacy', 'bhatbhateni',
                 'school', 'college', 'gym', 'public_transport',
                 'police_station', 'pashupatinath', 'boudhanath',
                 'atm', 'hotel', 'restaurant', 'banquet', 'ring_road']

for col in distance_cols:
    df_new[col + '_m'] = df_new[col].apply(convert_to_meters)

# Cap unrealistic values
distance_caps = {
    'hospital_m': 15000, 'airport_m': 30000, 'pharmacy_m': 10000,
    'bhatbhateni_m': 15000, 'school_m': 10000, 'college_m': 15000,
    'gym_m': 10000, 'public_transport_m': 5000, 'police_station_m': 15000,
    'pashupatinath_m': 30000, 'boudhanath_m': 30000, 'atm_m': 5000,
    'hotel_m': 15000, 'restaurant_m': 10000, 'banquet_m': 15000,
    'ring_road_m': 30000
}
for col, cap in distance_caps.items():
    df_new.loc[df_new[col] > cap, col] = np.nan

print(f"\n✅ All distance columns converted and capped")
print(f"\nFinal shape: {df_new.shape}")
print(f"Nulls summary:")
nulls = df_new.isnull().sum()
print(nulls[nulls > 0].sort_values(ascending=False).head(20))

Shape after dropping useless cols: (2328, 39)
total_price nulls: 3
Price range: 200,000 – 900,000,000

Rows removed by price filter: 138
Remaining rows: 2190

land_size_aana nulls: 1
Range: 1.0 – 50.0 aana

built_up_sqft nulls: 467
Range: 2 – 10000 sqft

house_age nulls: 367
Range: 0.0 – 61.0 years

road_width_feet nulls: 7

road_type distribution:
road_type
High Access    1643
Low Access      535
Name: count, dtype: int64

✅ All distance columns converted and capped

Final shape: (2190, 61)
Nulls summary:
dimension           1395
banquet_m            903
banquet              876
hotel_m              873
atm_m                864
hotel                863
restaurant_m         863
atm                  850
restaurant           846
gym_m                828
gym                  800
police_station_m     691
police_station       668
ring_road_m          663
ring_road            662
college_m            633
college              627
bhatbhateni_m        531
pharmacy_m           530
bhatbhateni  

In [7]:
from sklearn.impute import KNNImputer

# ─────────────────────────────────────────
# STEP 10 — DROP USELESS AND RAW COLUMNS
# ─────────────────────────────────────────
drop_cols = [
    # High null low value amenities
    'dimension', 'banquet', 'hotel', 'atm', 'restaurant', 'gym',
    'banquet_m', 'hotel_m', 'atm_m', 'restaurant_m', 'gym_m',
    # Raw string columns already parsed
    'police_station', 'ring_road', 'college', 'bhatbhateni',
    'pharmacy', 'boudhanath', 'pashupatinath', 'public_transport',
    'hospital', 'airport', 'school',
    'road_access', 'built_up_area', 'year_built',
    'year_built_ad', 'property_area',
    # Not useful for modeling
    'date_posted', 'views', 'negotiable',
    'total_price_raw', 'landmark'
]
# Only drop columns that exist
drop_cols = [c for c in drop_cols if c in df_new.columns]
df_new.drop(columns=drop_cols, inplace=True)
print(f"Shape after dropping: {df_new.shape}")

# ─────────────────────────────────────────
# CATEGORY 7 — DROP CRITICAL NULL ROWS
# ─────────────────────────────────────────
before = len(df_new)
df_new.dropna(subset=['total_price', 'property_type'], inplace=True)
df_new.reset_index(drop=True, inplace=True)
print(f"Rows dropped (missing target): {before - len(df_new)}")

# ─────────────────────────────────────────
# CATEGORY 3 — CONDITIONAL FILLING
# ─────────────────────────────────────────
df_new['furnishing']  = df_new['furnishing'].fillna('Unfurnished')
df_new['road_type']   = df_new['road_type'].fillna('High Access')
df_new['property_face'] = df_new['property_face'].str.title()
print(f"✅ furnishing, road_type filled")

# ─────────────────────────────────────────
# FIX property_face
# ─────────────────────────────────────────
df_new['property_face'] = df_new['property_face'].fillna(
    df_new['property_face'].mode()[0])
print(f"✅ property_face standardized")

# ─────────────────────────────────────────
# CATEGORY 2 — LOGIC BASED DEDUCTION
# neighborhood → municipality → ward_no
# ─────────────────────────────────────────
# Fix neighborhood nulls first
df_new['neighborhood'] = df_new.groupby('district')['neighborhood']\
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))

# Municipality using neighborhood mode
df_new['municipality'] = df_new.groupby('neighborhood')['municipality']\
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
df_new['municipality'] = df_new.groupby('district')['municipality']\
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))

# ward_no — convert to numeric first then fill
df_new['ward_no'] = pd.to_numeric(df_new['ward_no'], errors='coerce')
df_new['ward_no'] = df_new.groupby('neighborhood')['ward_no']\
    .transform(lambda x: x.fillna(x.mode()[0] if len(x.mode()) > 0 else np.nan))
df_new['ward_no'] = df_new.groupby('municipality')['ward_no']\
    .transform(lambda x: x.fillna(x.dropna().median() if not x.dropna().empty else np.nan))
df_new['ward_no'] = df_new['ward_no'].fillna(df_new['ward_no'].median())
df_new['ward_no'] = df_new['ward_no'].astype(int)
print(f"✅ neighborhood, municipality, ward_no filled")

# ─────────────────────────────────────────
# CATEGORY 4 — GROUPED MEDIAN IMPUTATION
# ─────────────────────────────────────────

# house_age — neighborhood median
df_new['house_age'] = df_new.groupby('neighborhood')['house_age']\
    .transform(lambda x: x.fillna(x.median()))
df_new['house_age'] = df_new['house_age'].fillna(df_new['house_age'].median())
df_new['house_age'] = df_new['house_age'].clip(upper=60)

# road_width_feet — neighborhood median
df_new['road_width_feet'] = df_new.groupby('neighborhood')['road_width_feet']\
    .transform(lambda x: x.fillna(x.median()))
df_new['road_width_feet'] = df_new['road_width_feet']\
    .fillna(df_new['road_width_feet'].median())

# land_size_aana — neighborhood median
df_new['land_size_aana'] = df_new.groupby('neighborhood')['land_size_aana']\
    .transform(lambda x: x.fillna(x.median()))
df_new['land_size_aana'] = df_new['land_size_aana']\
    .fillna(df_new['land_size_aana'].median())

# built_up_sqft — neighborhood + land_size bin grouped median
df_new['land_size_bin'] = pd.cut(df_new['land_size_aana'],
                                  bins=[0, 3, 5, 8, 15, 100],
                                  labels=['tiny','small','medium','large','xlarge'])
df_new['built_up_sqft'] = df_new.groupby(['neighborhood','land_size_bin'])['built_up_sqft']\
    .transform(lambda x: x.fillna(x.median()))
df_new['built_up_sqft'] = df_new.groupby('neighborhood')['built_up_sqft']\
    .transform(lambda x: x.fillna(x.median()))
df_new['built_up_sqft'] = df_new['built_up_sqft']\
    .fillna(df_new['built_up_sqft'].median())
df_new.drop(columns=['land_size_bin'], inplace=True)

# Fix built_up_sqft below 100
mask = df_new['built_up_sqft'] < 100
df_new.loc[mask, 'built_up_sqft'] = df_new.loc[mask, 'neighborhood']\
    .map(df_new.groupby('neighborhood')['built_up_sqft']
    .apply(lambda x: x[x >= 100].median()))
df_new['built_up_sqft'] = df_new['built_up_sqft']\
    .fillna(df_new['built_up_sqft'].median())

print(f"✅ house_age, road_width_feet, land_size_aana, built_up_sqft filled")

# ─────────────────────────────────────────
# NEW COLUMNS — bedrooms, bathrooms etc.
# Fill with neighborhood median
# Why: bedrooms in Budhanilkantha differ from Thamel
# ─────────────────────────────────────────
stat_cols = ['bedrooms', 'bathrooms', 'kitchens',
             'living_rooms', 'parking', 'total_floors']

for col in stat_cols:
    # Neighborhood median first
    df_new[col] = df_new.groupby('neighborhood')[col]\
        .transform(lambda x: x.fillna(x.median()))
    # Global median as fallback
    df_new[col] = df_new[col].fillna(df_new[col].median())
    # Round to nearest 0.5 — floors come in 0.5 increments
    df_new[col] = (df_new[col] * 2).round() / 2
    print(f"  ✅ {col} filled — range: {df_new[col].min()} – {df_new[col].max()}")

# ─────────────────────────────────────────
# CATEGORY 5 — KNN FOR DISTANCES
# ─────────────────────────────────────────
dist_knn_cols = ['hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m',
                 'school_m', 'college_m', 'public_transport_m',
                 'police_station_m', 'boudhanath_m', 'ring_road_m']

# Clip to 50m minimum before KNN
for col in dist_knn_cols:
    df_new[col] = df_new[col].clip(lower=50)

# Drop pashupatinath_m — high correlation with airport_m
if 'pashupatinath_m' in df_new.columns:
    df_new.drop(columns=['pashupatinath_m'], inplace=True)
    print(f"\n✅ Dropped pashupatinath_m (multicollinearity)")

knn_imputer = KNNImputer(n_neighbors=5)
df_new[dist_knn_cols] = knn_imputer.fit_transform(df_new[dist_knn_cols])
print(f"✅ Distance columns KNN imputed")

# ─────────────────────────────────────────
# MUNICIPALITY STANDARDIZATION
# Same map as previous cleaning
# ─────────────────────────────────────────
municipality_map = {
    'Kathmandu': 'Kathmandu', 'KMC': 'Kathmandu', 'Kmc': 'Kathmandu',
    'Kathmandu Metropolitan': 'Kathmandu', 'Kathmandu metropolitan': 'Kathmandu',
    'Kathmandu Metropolitan City': 'Kathmandu', 'Kathmandu Metropolitan city': 'Kathmandu',
    'Kathmandu mahanagar': 'Kathmandu', 'Kathamandu': 'Kathmandu',
    'kathmandu': 'Kathmandu', 'kathmandu metropolitan': 'Kathmandu',
    'karhmandu': 'Kathmandu', 'Kathmy': 'Kathmandu',
    'Budhanilkantha': 'Budhanilkantha', 'Budanilkantha': 'Budhanilkantha',
    'Budhanilkanta': 'Budhanilkantha', 'Budhanilakantha': 'Budhanilkantha',
    'budhanilkantha': 'Budhanilkantha', 'budhanilkanta': 'Budhanilkantha',
    'Budhanilktha': 'Budhanilkantha', 'Budhanilkatha': 'Budhanilkantha',
    'Budhanikatha': 'Budhanilkantha', 'Budhaknilkantha': 'Budhanilkantha',
    'Budhanilkntha': 'Budhanilkantha', 'Bhudhanilkatha': 'Budhanilkantha',
    'Bidanilkantha': 'Budhanilkantha', 'Budanilkntha': 'Budhanilkantha',
    'Budanilknntha': 'Budhanilkantha', 'Budanillantha': 'Budhanilkantha',
    'Budhanilkantha Nagarpalika': 'Budhanilkantha',
    'Budhanilkantha Mahanagarpalika': 'Budhanilkantha',
    'Budhanilkantha Maha Nagarpalika': 'Budhanilkantha',
    'Mahalaxmi': 'Mahalaxmi', 'mahalaxmi': 'Mahalaxmi',
    'Mahalaxmi Muncipality': 'Mahalaxmi', 'Mahalakxmi': 'Mahalaxmi',
    'Mahalakshmi': 'Mahalaxmi', 'Mahalaxmisthan': 'Mahalaxmi',
    'Lalitpur': 'Lalitpur', 'lalitpur': 'Lalitpur', 'LMC': 'Lalitpur',
    'Lalitpur MPC': 'Lalitpur', 'Lalitpur Metropolitan City': 'Lalitpur',
    'Lalitpur mahanagapalika': 'Lalitpur', 'Metropolitan Lalitpur': 'Lalitpur',
    'Lalpitpur': 'Lalitpur', 'Laly': 'Lalitpur', 'jhamsikhel': 'Lalitpur',
    'Dholahiti': 'Lalitpur', 'Kusunti': 'Lalitpur',
    'Bhaisepati': 'Lalitpur', 'Sainbu': 'Lalitpur',
    'Tokha': 'Tokha', 'Tokha Nagarpalika': 'Tokha',
    'Tokha Municipality': 'Tokha', 'Thankot': 'Tokha',
    'Kageshwori Manohara': 'Kageshwori Manohara',
    'Kageswori Manohara': 'Kageshwori Manohara',
    'Kageshwori Manahora': 'Kageshwori Manohara',
    'Kageshwori Monahara': 'Kageshwori Manohara',
    'Kageshwori Monahora': 'Kageshwori Manohara',
    'Kageshwori Manohar': 'Kageshwori Manohara',
    'Kageshwori Manohora': 'Kageshwori Manohara',
    'Kageshwari Manohara': 'Kageshwori Manohara',
    'Kageshwari-Manohara': 'Kageshwori Manohara',
    'Kageshwori, Manohara': 'Kageshwori Manohara',
    'Kagashwori Manohara': 'Kageshwori Manohara',
    'Kagaeshwori Manahora': 'Kageshwori Manohara',
    'kageswori manohara': 'Kageshwori Manohara',
    'Kageswori mahonara':'Kageshwori Manohara',
    'Kageswori manohara': 'Kageshwori Manohara',
    'Kageshwori Manahara Municipality': 'Kageshwori Manohara',
    'Kageshowri Manohara': 'Kageshwori Manohara',
    'Suryabinayak': 'Suryabinayak', 'Suryavinayak': 'Suryabinayak',
    'Suryabinyak': 'Suryabinayak', 'Suryabinak': 'Suryabinayak',
    'Suryabnayak': 'Suryabinayak', 'suryabinayak': 'Suryabinayak',
    'Suryabinayak Municipality': 'Suryabinayak',
    'Suryabinayak Na. pa.': 'Suryabinayak',
    'Suryabinayak Na. Pa.': 'Suryabinayak',
    'Suryabinayak Nagarpalika': 'Suryabinayak',
    'Nagarjun': 'Nagarjun', 'Nagargun': 'Nagarjun',
    'Nagarjun Nagarpalika': 'Nagarjun', 'Ramkot': 'Nagarjun',
    'Baluwakhani': 'Nagarjun', 'Balambu': 'Nagarjun',
    'kalanki': 'Nagarjun', 'Syuchatar': 'Nagarjun',
    'Madhyapur Thimi': 'Madhyapur Thimi',
    'Madhyapur Thimi Municipality': 'Madhyapur Thimi',
    'Madyapur Thimi': 'Madhyapur Thimi', 'Madhyapur': 'Madhyapur Thimi',
    'Tikathali': 'Madhyapur Thimi', 'Balkot': 'Madhyapur Thimi',
    'Gokarneswor': 'Gokarneshwor', 'Gokarneshwor': 'Gokarneshwor',
    'Gokarneshwar': 'Gokarneshwor', 'Gokarnashwor': 'Gokarneshwor',
    'Gokenshowr': 'Gokarneshwor', 'gokarneswor': 'Gokarneshwor',
    'Boudha': 'Gokarneshwor', 'Baneshwor': 'Gokarneshwor',
    'Tarkeshwor': 'Tarkeshwor', 'Tarkeshwor Municipality': 'Tarkeshwor',
    'Tarkeshwor Nagarpalika': 'Tarkeshwor', 'Tarkeshwar': 'Tarkeshwor',
    'Tarakeshwar municipality': 'Tarkeshwor',
    'tarkeshwor municipality': 'Tarkeshwor',
    'Godawari': 'Godawari', 'godawari': 'Godawari',
    'Chandragiri': 'Chandragiri', 'chandragiri': 'Chandragiri',
    'Changunarayan': 'Changunarayan',
    'Kirtipur': 'Kirtipur', 'kirtipur': 'Kirtipur', 'Kritipur': 'Kirtipur',
    'Bhaktapur': 'Bhaktapur',
}
df_new['municipality'] = df_new['municipality'].replace(municipality_map)
# Remaining unmapped → district mode
df_new['municipality'] = df_new.groupby('district')['municipality']\
    .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown'))
print(f"\n✅ Municipality standardized: {df_new['municipality'].nunique()} unique")

# ─────────────────────────────────────────
# NEIGHBORHOOD ZONE CLEANING
# Same thresholds as previous cleaning
# ─────────────────────────────────────────
rare_neighborhoods = df_new['neighborhood'].value_counts()
rare_neighborhoods = rare_neighborhoods[rare_neighborhoods < 3].index.tolist()
rare_prices = df_new[df_new['neighborhood'].isin(rare_neighborhoods)]\
    .groupby('neighborhood')['total_price'].median()

def assign_zone(price):
    if price >= 75000000:  return 'Premium_Zone'
    elif price >= 48000000: return 'High_Zone'
    elif price >= 35000000: return 'Mid_Zone'
    elif price >= 26000000: return 'Budget_Zone'
    else:                   return 'Outskirt_Zone'

zone_map = {n: assign_zone(p) for n, p in rare_prices.items()}
df_new['neighborhood'] = df_new['neighborhood'].apply(
    lambda x: zone_map.get(x, x))

# Merge High_Zone if too small
if df_new['neighborhood'].value_counts().get('High_Zone', 0) < 5:
    df_new['neighborhood'] = df_new['neighborhood'].replace('High_Zone', 'Mid_Zone')

print(f"✅ Neighborhood cleaned: {df_new['neighborhood'].nunique()} unique")

# ─────────────────────────────────────────
# FINAL CHECKS
# ─────────────────────────────────────────
print(f"\n=== Final shape: {df_new.shape} ===")
print(f"Nulls: {df_new.isnull().sum().sum()}")
print(f"Duplicates: {df_new.duplicated().sum()}")
print(f"\nNew columns distribution:")
for col in stat_cols:
    print(f"  {col}: mean={df_new[col].mean():.1f}, "
          f"min={df_new[col].min()}, max={df_new[col].max()}")

Shape after dropping: (2190, 30)
Rows dropped (missing target): 0
✅ furnishing, road_type filled
✅ property_face standardized
✅ neighborhood, municipality, ward_no filled


C:\Users\DELL\AppData\Local\Temp\ipykernel_10952\4265382820.py:58: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .transform(lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan))
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\DELL\AppData\Local\Packages\PythonSoftware

✅ house_age, road_width_feet, land_size_aana, built_up_sqft filled
  ✅ bedrooms filled — range: 1.0 – 40.0
  ✅ bathrooms filled — range: 1.0 – 22.0
  ✅ kitchens filled — range: 1.0 – 8.0
  ✅ living_rooms filled — range: 1.0 – 8.0
  ✅ parking filled — range: 1.0 – 12.0
  ✅ total_floors filled — range: 0.0 – 275.0

✅ Dropped pashupatinath_m (multicollinearity)


C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\numpy\lib\_nanfunctions_impl.py:1241: RuntimeWarning: M

✅ Distance columns KNN imputed

✅ Municipality standardized: 17 unique
✅ Neighborhood cleaned: 142 unique

=== Final shape: (2190, 29) ===
Nulls: 0
Duplicates: 3

New columns distribution:
  bedrooms: mean=6.5, min=1.0, max=40.0
  bathrooms: mean=4.2, min=1.0, max=22.0
  kitchens: mean=1.9, min=1.0, max=8.0
  living_rooms: mean=1.9, min=1.0, max=8.0
  parking: mean=2.3, min=1.0, max=12.0
  total_floors: mean=3.0, min=0.0, max=275.0


In [8]:
# ─────────────────────────────────────────
# FIX OUTLIERS IN NEW COLUMNS
# ─────────────────────────────────────────

# Check suspicious values first
print("=== total_floors suspicious values ===")
print(df_new[df_new['total_floors'] > 10]['total_floors'].value_counts().head(10))

print("\n=== bedrooms suspicious values ===")
print(df_new[df_new['bedrooms'] > 15]['bedrooms'].value_counts().head(10))

print("\n=== bathrooms suspicious values ===")
print(df_new[df_new['bathrooms'] > 12]['bathrooms'].value_counts().head(10))

# Realistic caps for Nepal housing:
# total_floors: max 6 floors for residential — above 10 is error
# bedrooms: max 12 for large houses — above 15 is error
# bathrooms: max 10 — above 12 is error
# parking: max 10 — realistic for large properties

caps = {
    'total_floors': 10,   # 10 floor max — high-rise cap
    'bedrooms':     15,   # 15 bedroom max — very large house
    'bathrooms':    10,   # 10 bathroom max
    'parking':      10,   # 10 parking max
}

for col, cap in caps.items():
    over = (df_new[col] > cap).sum()
    if over > 0:
        print(f"\n⚠️  {col}: {over} rows above {cap} — capping with neighborhood median")
        # Replace with neighborhood median rather than just capping
        # Why: 275 floors is clearly wrong — neighborhood median is more accurate
        mask = df_new[col] > cap
        df_new.loc[mask, col] = df_new.loc[mask, 'neighborhood']\
            .map(df_new[~mask].groupby('neighborhood')[col].median())
        # Fill any remaining with global median
        df_new[col] = df_new[col].fillna(df_new[col].median())
        # Round to nearest 0.5
        df_new[col] = (df_new[col] * 2).round() / 2

# ─────────────────────────────────────────
# DROP DUPLICATES
# ─────────────────────────────────────────
before = len(df_new)
df_new.drop_duplicates(inplace=True)
df_new.reset_index(drop=True, inplace=True)
print(f"\n✅ Duplicates removed: {before - len(df_new)}")

# ─────────────────────────────────────────
# FINAL VERIFY
# ─────────────────────────────────────────
print(f"\n=== Final shape: {df_new.shape} ===")
print(f"Nulls: {df_new.isnull().sum().sum()}")
print(f"Duplicates: {df_new.duplicated().sum()}")
print(f"\nNew columns after fixing:")
for col in ['bedrooms', 'bathrooms', 'kitchens',
            'living_rooms', 'parking', 'total_floors']:
    print(f"  {col}: mean={df_new[col].mean():.1f}, "
          f"min={df_new[col].min()}, max={df_new[col].max()}")



=== total_floors suspicious values ===
total_floors
25.0     1
13.0     1
275.0    1
35.0     1
Name: count, dtype: int64

=== bedrooms suspicious values ===
bedrooms
20.0    20
16.0    17
17.0     5
22.0     5
19.0     4
18.0     3
28.0     2
15.5     2
32.0     1
40.0     1
Name: count, dtype: int64

=== bathrooms suspicious values ===
bathrooms
15.0    3
20.0    2
22.0    1
18.0    1
19.0    1
16.0    1
Name: count, dtype: int64

⚠️  total_floors: 4 rows above 10 — capping with neighborhood median

⚠️  bedrooms: 63 rows above 15 — capping with neighborhood median

⚠️  bathrooms: 16 rows above 10 — capping with neighborhood median

⚠️  parking: 2 rows above 10 — capping with neighborhood median

✅ Duplicates removed: 3

=== Final shape: (2187, 29) ===
Nulls: 0
Duplicates: 0

New columns after fixing:
  bedrooms: mean=6.2, min=1.0, max=15.0
  bathrooms: mean=4.1, min=1.0, max=10.0
  kitchens: mean=1.9, min=1.0, max=8.0
  living_rooms: mean=1.9, min=1.0, max=8.0
  parking: mean=2.3, mi

In [9]:
# ─────────────────────────────────────────
# SAVE
# ─────────────────────────────────────────
df_new.to_csv('cleaned_lalpurja_house_v2_after_cleaning.csv', index=False)
print(f"\n✅ Saved cleaned_lalpurja_house_v2_after_cleaning.csv")
print(f"✅ Columns: {list(df_new.columns)}")


✅ Saved cleaned_lalpurja_house_v2_after_cleaning.csv
✅ Columns: ['district', 'property_type', 'property_face', 'road_type', 'furnishing', 'municipality', 'ward_no', 'neighborhood', 'bedrooms', 'kitchens', 'bathrooms', 'living_rooms', 'parking', 'total_floors', 'total_price', 'land_size_aana', 'built_up_sqft', 'house_age', 'road_width_feet', 'hospital_m', 'airport_m', 'pharmacy_m', 'bhatbhateni_m', 'school_m', 'college_m', 'public_transport_m', 'police_station_m', 'boudhanath_m', 'ring_road_m']
